Importing The Dependencies

In [43]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier

Data Collection & Preprocessing

In [44]:
# load the clean data from csv to a pandas DataFrame
df = pd.read_csv("../dataset/cleaned data/clean_credit_risk_dataset.csv")

In [45]:
df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,21,9600,own,5.0,education,b,1000,11.14,0,0.10,n,2
1,25,9600,mortgage,1.0,medical,c,5500,12.87,1,0.57,n,3
2,23,65500,rent,4.0,medical,c,35000,15.23,1,0.53,n,2
3,24,54400,rent,8.0,medical,c,35000,14.27,1,0.55,y,4
4,21,9900,own,2.0,venture,a,2500,7.14,1,0.25,n,2


In [46]:
df['loan_status'].value_counts()

loan_status
0    25321
1     7088
Name: count, dtype: int64

0 -> Paid (Non-Default Value)

1 -> Not Paid (Default Value)

Separating Features and Target

In [47]:
# separate features (X) and target (Y)
X = df.drop('loan_status', axis=1)
Y = df['loan_status']

In [48]:
print(X)

       person_age  person_income person_home_ownership  person_emp_length  \
0              21           9600                   own                5.0   
1              25           9600              mortgage                1.0   
2              23          65500                  rent                4.0   
3              24          54400                  rent                8.0   
4              21           9900                   own                2.0   
...           ...            ...                   ...                ...   
32404          57          53000              mortgage                1.0   
32405          54         120000              mortgage                4.0   
32406          65          76000                  rent                3.0   
32407          56         150000              mortgage                5.0   
32408          66          42000                  rent                2.0   

           loan_intent loan_grade  loan_amnt  loan_int_rate  \
0           

In [49]:
print(Y)

0        0
1        1
2        1
3        1
4        1
        ..
32404    0
32405    0
32406    1
32407    0
32408    0
Name: loan_status, Length: 32409, dtype: int64


Splitting the data into Training and Test Data

In [50]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=3)

In [51]:
print(X.shape, X_train.shape, X_test.shape)

(32409, 11) (25927, 11) (6482, 11)


Label Encoder

In [52]:
# load the label encoder
label_encoder = LabelEncoder()

In [53]:
# fit and transform training and test data
X_train['cb_person_default_on_file'] = label_encoder.fit_transform(X_train['cb_person_default_on_file'])
X_test['cb_person_default_on_file'] = label_encoder.fit_transform(X_test['cb_person_default_on_file'])

In [54]:
X_train.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
27771,27,42504,rent,2.0,personal,b,7200,10.59,0.17,0,9
5095,25,43500,mortgage,2.0,homeimprovement,a,9000,9.63,0.21,0,4
29263,44,26500,mortgage,5.0,debtconsolidation,a,6200,6.03,0.23,0,16
10103,24,55000,rent,5.0,venture,a,9500,9.63,0.17,0,4
26849,31,120000,mortgage,8.0,education,a,20000,7.49,0.17,0,6


One-Hot Encoder

In [55]:
# Define Categories
category_cols = ['person_home_ownership','loan_intent','loan_grade']

# initialize    andfit only on training data
encoder = OneHotEncoder(sparse_output = False, handle_unknown='ignore')
encoder.fit(X_train[category_cols])

# Transform  both 
train_encoded = encoder.transform(X_train[category_cols])
test_encoded = encoder.transform(X_test[category_cols])

# Reverting it to dataframe
train_encoded_df =pd.DataFrame(train_encoded, columns = encoder.get_feature_names_out(category_cols), index = X_train.index)
test_encoded_df =pd.DataFrame(test_encoded, columns = encoder.get_feature_names_out(category_cols), index = X_test.index)

# drop old string columns and the new column with value 0 and 1
X_train_final = pd.concat([X_train.drop(category_cols, axis=1), train_encoded_df], axis=1)
X_test_final = pd.concat([X_test.drop(category_cols, axis=1), test_encoded_df], axis=1)

In [56]:
print(train_encoded_df)
print(test_encoded_df)

       person_home_ownership_mortgage  person_home_ownership_other  \
27771                             0.0                          0.0   
5095                              1.0                          0.0   
29263                             1.0                          0.0   
10103                             0.0                          0.0   
26849                             1.0                          0.0   
...                               ...                          ...   
28928                             0.0                          0.0   
3818                              0.0                          0.0   
24417                             0.0                          0.0   
12925                             0.0                          0.0   
23292                             1.0                          0.0   

       person_home_ownership_own  person_home_ownership_rent  \
27771                        0.0                         1.0   
5095                         0.

Data Standardization

In [57]:
# Define Numerical Value
numerical_cols = ['person_age','person_income','person_emp_length','loan_amnt','loan_int_rate','loan_percent_income','cb_person_cred_hist_length']

scaler = StandardScaler()

# fit only on Training Data
scaler.fit(X_train_final[numerical_cols])

# Transform both sets
X_train_final_scale = X_train_final.copy()
X_test_final_scale = X_test_final.copy()

X_train_final_scale[numerical_cols] = scaler.transform(X_train_final[numerical_cols])
X_test_final_scale[numerical_cols] = scaler.transform(X_test_final[numerical_cols])

print("Data is now encoded and scaled. Preview of first 5 rows")
X_train_final.head()

Data is now encoded and scaled. Preview of first 5 rows


,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,person_home_ownership_mortgage,person_home_ownership_other,...,loan_intent_medical,loan_intent_personal,loan_intent_venture,loan_grade_a,loan_grade_b,loan_grade_c,loan_grade_d,loan_grade_e,loan_grade_f,loan_grade_g
27771,27,42504,2.0,7200,10.59,0.17,0,9,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
5095,25,43500,2.0,9000,9.63,0.21,0,4,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
29263,44,26500,5.0,6200,6.03,0.23,0,16,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
10103,24,55000,5.0,9500,9.63,0.17,0,4,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
26849,31,120000,8.0,20000,7.49,0.17,0,6,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


**Model Selection**

In [58]:
# list of models

models = [LogisticRegression(max_iter = 10000), RandomForestClassifier()]

def compare_models_cross_validation():
    for model in models:
    
        cv_score = cross_val_score(model , X_train_final_scale ,Y_train , cv=5)
        mean_accuracy = (sum(cv_score)/len(cv_score))* 100
        mean_accuracy = round(mean_accuracy, 2)

        print(f"Cross Validation Accuracies for the {model} = {cv_score}")
        print(f'Accuracy Score of the {mean_accuracy} %.')
        print('-' * 150)

        
compare_models_cross_validation()

Cross Validation Accuracies for the LogisticRegression(max_iter=10000) = [0.85576552 0.86887775 0.86075217 0.87020251 0.86711668]
Accuracy Score of the 86.45 %.
------------------------------------------------------------------------------------------------------------------------------------------------------
Cross Validation Accuracies for the RandomForestClassifier() = [0.92749711 0.93655997 0.93172613 0.93346191 0.93577628]
Accuracy Score of the 93.3 %.
------------------------------------------------------------------------------------------------------------------------------------------------------


For this dataset, **Random Forest Classifier** has the highest accuracy value with default hyperparameter values

Now, Comparing the models with different Hyperparameter values using GridSearchCV

In [59]:
# creating a dictionary that contains hyperparameter values for the above mentioned models

model_hyperparameters= {
    'log_reg_hyperparameters': {
        'C' :[1,5,10,20]
        },

    'random_forest_hyperparameters' : {
        'n_estimators' : [10, 20, 50,100]
    }
}

print(model_hyperparameters)
print(model_hyperparameters.keys())

{'log_reg_hyperparameters': {'C': [1, 5, 10, 20]}, 'random_forest_hyperparameters': {'n_estimators': [10, 20, 50, 100]}}
dict_keys(['log_reg_hyperparameters', 'random_forest_hyperparameters'])


In [60]:
model_keys = list(model_hyperparameters.keys())
print(model_keys)

type(model_keys)

['log_reg_hyperparameters', 'random_forest_hyperparameters']


list

Applying GridSearchCV 

In [61]:
def ModelSelection(list_of_models, hyperparameters_dictionary):
    result = []

    i = 0

    for model in list_of_models:
        key = model_keys[i]

        params =hyperparameters_dictionary[key]

        i+=1

        print(model)
        print(params)
        print('-' * 180)

        classifier = GridSearchCV(model, params, cv=5)

        # fitting the processed data to classifier
        classifier.fit(X_train_final_scale, Y_train)

        result.append({
            'model used' : model,
            'highest score' : classifier.best_score_,
            'best hyperparameters' : classifier.best_params_
        })

    result_dataframe = pd.DataFrame(result, columns= ['model used','highest score','best hyperparameters'])

    return result_dataframe

In [62]:
ModelSelection(models, model_hyperparameters)

LogisticRegression(max_iter=10000)
{'C': [1, 5, 10, 20]}
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
RandomForestClassifier()
{'n_estimators': [10, 20, 50, 100]}
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


,model used,highest score,best hyperparameters
0,LogisticRegression(max_iter=10000),0.864929,{'C': 20}
1,RandomForestClassifier(),0.933159,{'n_estimators': 100}


Random Forest Classifier has the highest accuracy when n_estimators is 100.